# Lab 3 — 이벤트 품질과 이상 감지 (40분)
**목표:** 같은 이벤트 재전송, 순서 역전, 닫힘 이벤트 미도착을 각각 검증합니다.
Lab 2의 `resource.href/unitId/component`, `eventTime`, `payload.doorState`를 그대로 사용합니다.
원본 4,959건은 유지하고 추가 시나리오는 `SYNTHETIC_LAB3`로 구분합니다.

위에서 아래로 **Run All**. 순서: 샘플 5분 → 멱등성 10분 → 최신 상태 10분 → timer 10분 → 리뷰 5분.
전용 DB `hrm_workshop_lab3`에 쓰기 작업을 수행합니다. MCP 읽기 전용 설정과 별도로 PyMongo를 사용합니다.

In [1]:
# 공통 모듈을 찾고 Lab별 독립 DB에 연결합니다. URI/비밀번호는 출력하지 않습니다.
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
from copy import deepcopy
from collections import Counter
from IPython.display import display
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab2/events.ejson").exists())
sys.path.insert(0, str(root))
from workshop_utils import load_samples, connect, insert_once, table, ensure_search_index
events, devices = load_samples()
client, db = connect("hrm_workshop_lab3")
UTC = timezone.utc
print("기기", len(devices), "/ 이벤트", len(events), "/ 종류", dict(Counter(e["eventType"] for e in events)))
assert len(devices) == 24 and len(events) == 4959
display(table(events[:2]))

연결 성공 / MongoDB 8.0.34 / DB hrm_workshop_lab3
기기 24 / 이벤트 4959 / 종류 {'mode_snapshot': 3360, 'door_open': 800, 'door_close': 799}


,_id,deviceId,locationId,eventType,eventTime,receivedAt,schemaVersion,datasetTag,resource.href,resource.unitId,resource.component,context.modelCode,context.regionCode,context.countryCode,context.timeZone,payload.modes,notification.source,notification.capabilities
0,LAB2-D04-0-mode-0,LAB2-D04,LAB2-L04,mode_snapshot,2026-10-04 21:00:00+00:00,2026-10-04 21:00:00.300000+00:00,1,HRM_LAB2_DOOR_V1,/lab2/mode/0,0,main,LAB2-MODEL-1,DEMO-SEOUL,KR,Asia/Seoul,[MODE_01],SYNTHETIC_LAB2,[execute]
1,LAB2-D08-0-mode-0,LAB2-D08,LAB2-L08,mode_snapshot,2026-10-04 21:00:00+00:00,2026-10-04 21:00:00.300000+00:00,1,HRM_LAB2_DOOR_V1,/lab2/mode/0,0,main,LAB2-MODEL-2,DEMO-SEOUL,KR,Asia/Seoul,[MODE_01],SYNTHETIC_LAB2,[execute]


## 1. 결정적 이벤트 ID와 재전송
`_id`는 발행자가 재전송에도 유지해야 합니다. 다른 ID의 업무상 중복은 별도 문제입니다.

In [2]:
# 같은 원본을 두 번 적재해도 _id 기준 이력 건수는 4,959건입니다.
history = db.events
insert_once(history, events)
insert_once(history, events)
original_count = history.count_documents({"datasetTag": "HRM_LAB2_DOOR_V1"})
assert original_count == len(events)
display(table(history.aggregate([
    {"$match": {"datasetTag": "HRM_LAB2_DOOR_V1"}},
    {"$group": {"_id": "$eventType", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
])))
print("재전송 검증 OK:", original_count)

,_id,count
0,door_close,799
1,door_open,800
2,mode_snapshot,3360


재전송 검증 OK: 4959


## 2. 리소스별 최신 상태 — 늦게 온 OPEN이 CLOSED를 되돌리지 않게 하기
신규 상태는 **키만으로 생성**, 갱신은 **새로운 eventTime일 때만, upsert=False**로 처리합니다.
이 예제는 리소스 전체 상태 이벤트와 단일 순차 consumer를 가정합니다.
같은 시각의 서로 다른 이벤트는 현재 정책상 첫 값 유지입니다. 운영에서는 생산자 버전/sequence를 합의하세요.
부분 payload에는 필드별 버전이 필요하며 Kafka offset은 partition 사이의 순서를 비교하지 못합니다.

In [3]:
# 서로 다른 문/모드 리소스를 섞지 않도록 기기와 전체 리소스 경로로 상태 키를 만듭니다.
import json
from pymongo.errors import DuplicateKeyError
states = db.current_state
def state_key(e):
    r = e["resource"]
    return json.dumps([e["deviceId"], r["href"], r["unitId"], r["component"]], separators=(",", ":"))

def process_event(e):
    """이력 저장과 상태 갱신 모두 멱등하게 수행하는 순차 처리 예제입니다."""
    insert_once(history, [e])
    key = state_key(e)
    # 신규 키를 먼저 준비합니다. 시간 조건과 upsert를 섞지 않습니다.
    try:
        states.update_one({"_id": key}, {"$setOnInsert": {
            "deviceId": e["deviceId"], "resource": e["resource"],
            "eventTime": datetime(1970, 1, 1, tzinfo=UTC)}}, upsert=True)
    except DuplicateKeyError:
        pass  # 동시 최초 생성 시 이미 만들어진 상태를 아래 조건부 갱신에서 사용합니다.
    previous = states.find_one({"_id": key})
    values = {"eventTime": e["eventTime"], "lastEventId": e["_id"], "payload": e["payload"]}
    if "doorState" in e["payload"]:
        values["doorState"] = e["payload"]["doorState"]
        # 같은 열림 구간의 반복 OPEN에는 처음 열린 시각과 이벤트 ID를 유지합니다.
        already_open = previous.get("doorState") == "open"
        values["openSince"] = previous.get("openSince") if already_open else e["eventTime"]
        values["openEventId"] = previous.get("openEventId") if already_open else e["_id"]
        if values["doorState"] == "closed":
            values["openSince"] = None
            values["openEventId"] = None
    # 읽은 버전도 조건에 포함하여 다른 consumer가 상태를 바꾼 경우 덮어쓰지 않습니다.
    result = states.update_one({"_id": key, "eventTime": previous["eventTime"],
                               "$expr": {"$lt": ["$eventTime", e["eventTime"]]}}, {"$set": values})
    # matched=0에는 stale/재전송뿐 아니라 경합도 포함됩니다. 병렬 처리기는 재조회/재시도해야 합니다.
    return result.modified_count

# 순차 처리 실습은 원본 도어 이벤트 12건만 재현합니다. 전체 이력은 앞 셀에서 적재했습니다.
replay = sorted((e for e in events if e["eventType"].startswith("door_")),
                key=lambda x: (x["eventTime"], x["_id"]))[:12]
for e in replay:
    process_event(e)
display(table(states.find({"doorState": "open"}, {"payload": 0}).limit(5)))

""


In [4]:
# Lab2의 실제 OPEN/CLOSE 쌍을 복사해 재전송과 순서 역전을 재현합니다.
closed = next(e for e in events if e["eventType"] == "door_close")
opened = next(e for e in events if e["_id"] == closed["payload"]["openEventId"])
def fixture(source, event_id, device_id, timestamp):
    e = deepcopy(source)
    e.update(_id=event_id, deviceId=device_id, eventTime=timestamp,
             receivedAt=timestamp + timedelta(milliseconds=300), datasetTag="SYNTHETIC_LAB3")
    return e
base = datetime(2026, 10, 12, 10, 0, tzinfo=UTC)
open_demo = fixture(opened, "lab3-order-open", "LAB3-ORDER", base)
close_demo = fixture(closed, "lab3-order-close", "LAB3-ORDER", base + timedelta(seconds=60))
close_demo["payload"].update(openEventId=open_demo["_id"], openedAt=base)
process_event(open_demo)
process_event(close_demo)
process_event(open_demo)  # 늦게 도착한 과거 OPEN은 무시됩니다.
observed = states.find_one({"_id": state_key(open_demo)})
assert observed["doorState"] == "closed" and observed["lastEventId"] == close_demo["_id"]
assert history.count_documents({"_id": open_demo["_id"]}) == 1
print("중복·순서 역전 검증 OK")

중복·순서 역전 검증 OK


## 3. 닫힘 이벤트가 없어도 180초를 평가하는 timer
고정 NOW에 대해 179초 제외, 180·181초 포함을 검증합니다.
알림 키는 상태 키+규칙 버전+열림 이벤트 ID입니다. 늦게 온 CLOSE는 같은 구간의 알림을 해소합니다.
아래는 교육용 순차 tick이며, 운영의 조회/알림 사이 경합은 트랜잭션 또는 후속 reconcile로 처리합니다.

In [5]:
# timer는 OPEN만 조회하므로 partial index로 불필요한 CLOSED 상태를 제외합니다.
alerts = db.anomaly_results
states.create_index([("openSince", 1)], name="open_timer",
                    partialFilterExpression={"doorState": "open"})
NOW = base + timedelta(minutes=3)
for duration in (179, 180, 181):
    e = fixture(opened, f"lab3-boundary-{duration}", f"LAB3-TIMER-{duration}", NOW - timedelta(seconds=duration))
    process_event(e)

def timer_tick(now):
    """고정 평가 시각으로 후보를 재확인하고 같은 열림 구간의 알림을 한 번만 기록합니다."""
    query = {"doorState": "open", "openSince": {"$lte": now - timedelta(seconds=180)},
             "deviceId": {"$in": [f"LAB3-TIMER-{s}" for s in (179, 180, 181)]}}
    for candidate in states.find(query):
        # 후보 추출 이후 닫힌 상태로 바뀌었으면 알림을 만들지 않습니다.
        current = states.find_one({"_id": candidate["_id"], "doorState": "open",
                                   "openEventId": candidate["openEventId"]})
        if current is None:
            continue
        alert_id = candidate["_id"] + "|door180-v1|" + candidate["openEventId"]
        insert_once(alerts, [{"_id": alert_id, "deviceId": candidate["deviceId"],
                              "openEventId": candidate["openEventId"], "status": "active",
                              "detectedAt": now, "durationSeconds": (now - candidate["openSince"]).total_seconds()}])
timer_tick(NOW)
timer_tick(NOW)
expected_ids = {"LAB3-TIMER-180", "LAB3-TIMER-181"}
actual = list(alerts.find({"deviceId": {"$in": [f"LAB3-TIMER-{s}" for s in (179, 180, 181)]}}))
assert {a["deviceId"] for a in actual} == expected_ids and len(actual) == 2
display(table(actual))

,_id,detectedAt,deviceId,durationSeconds,openEventId,status
0,"[""LAB3-TIMER-181"",""/lab2/door/0"",""0"",""main""]|d...",2026-10-12 10:03:00+00:00,LAB3-TIMER-181,181.0,lab3-boundary-181,active
1,"[""LAB3-TIMER-180"",""/lab2/door/0"",""0"",""main""]|d...",2026-10-12 10:03:00+00:00,LAB3-TIMER-180,180.0,lab3-boundary-180,active


In [6]:
# 181초 구간의 CLOSE가 늦게 도착하면 그 열림 이벤트의 알림만 해소합니다.
late_close = fixture(closed, "lab3-timer-late-close", "LAB3-TIMER-181", NOW + timedelta(seconds=1))
late_close["payload"].update(openEventId="lab3-boundary-181", openedAt=NOW - timedelta(seconds=181))
process_event(late_close)
current = states.find_one({"_id": state_key(late_close)})
if current["doorState"] == "closed":
    alerts.update_many({"deviceId": late_close["deviceId"], "openEventId": late_close["payload"]["openEventId"]},
                       {"$set": {"status": "resolved", "resolvedAt": late_close["eventTime"]}})
assert alerts.find_one({"deviceId": "LAB3-TIMER-181"})["status"] == "resolved"
# explain을 통해 timer 인덱스 후보와 검사량을 직접 확인합니다.
plan = db.command("explain", {"find": states.name,
                  "filter": {"doorState": "open", "openSince": {"$lte": NOW - timedelta(seconds=180)}}},
                  verbosity="executionStats")
display(plan["executionStats"])
client.close()  # 노트북 실습 종료 시 재사용하던 연결을 정리합니다.

{'executionSuccess': True,
 'nReturned': 1,
 'executionTimeMillis': 1,
 'totalKeysExamined': 1,
 'totalDocsExamined': 1,
 'executionStages': {'isCached': False,
  'stage': 'FETCH',
  'nReturned': 1,
  'executionTimeMillisEstimate': 0,
  'works': 2,
  'advanced': 1,
  'needTime': 0,
  'needYield': 0,
  'saveState': 0,
  'restoreState': 0,
  'isEOF': 1,
  'docsExamined': 1,
  'alreadyHasObj': 0,
  'inputStage': {'stage': 'IXSCAN',
   'nReturned': 1,
   'executionTimeMillisEstimate': 0,
   'works': 2,
   'advanced': 1,
   'needTime': 0,
   'needYield': 0,
   'saveState': 0,
   'restoreState': 0,
   'isEOF': 1,
   'keyPattern': {'openSince': 1},
   'indexName': 'open_timer',
   'isMultiKey': False,
   'multiKeyPaths': {'openSince': []},
   'isUnique': False,
   'isSparse': False,
   'isPartial': True,
   'indexVersion': 2,
   'direction': 'forward',
   'indexBounds': {'openSince': ['[new Date(-9223372036854775808), new Date(1791799200000)]']},
   'keysExamined': 1,
   'seeks': 1,
   'dupsT

## 리뷰
- 이력 저장 직후 재시작되면 상태 갱신을 다시 수행해야 하는 이유는?
- 서로 다른 ID로 발행된 동일 업무 이벤트는 어떻게 판정할까요?
- 부분 payload·동일 시각·병렬 consumer에서 필요한 추가 버전 계약을 기록하세요.
- 배치 bulk_write는 여러 문서의 트랜잭션이 아닙니다.

[조건부 갱신](https://www.mongodb.com/docs/manual/reference/method/db.collection.updateOne/) ·
[원자성](https://www.mongodb.com/docs/manual/core/write-operations-atomicity/)

## Kafka Connect 연결 가이드 — 준비와 연결 흐름

이 섹션은 Kafka 환경에서 실습을 확장하는 **가이드**입니다. 아래 코드는 Markdown 예시이므로
Run All로 Kafka 등록·발행이 자동 실행되지 않습니다. 실제 환경의 주소·인증을 채운 후 별도 셀/터미널에서 실행하세요.
현재 저장된 실행 결과는 PyMongo 직접 적재 결과입니다.

```text
노트북 Producer ── bootstrap.servers ──▶ Kafka Broker / 토픽
                                           │
                         Kafka Connect Worker (MongoDB Sink plugin)
                                           │ connection.uri
                                           ▼
                                     MongoDB Atlas

노트북 ── Connect REST API (:8083 예시) ──▶ Connector 등록·상태 확인
```

### 1. 주소 세 가지를 구분합니다

| 주소 | 용도 | 교육용 예시 |
|---|---|---|
| Kafka bootstrap server | Producer/Consumer와 Connect worker가 Broker 접속 | `localhost:9092` |
| Connect REST URL | Connector 등록·설정 검증·상태 확인 | `http://localhost:8083` |
| MongoDB URI | Connect worker가 Atlas 접속 | `mongodb+srv://<user>:<password>@<cluster>/` |

Docker 내부 worker의 `localhost`는 worker 컨테이너 자신입니다. Broker의 advertised.listeners는
노트북과 worker 각각에서 접근 가능해야 합니다(예: 호스트 `localhost:9092`, 내부 `kafka:29092`).
Atlas Network Access에는 **Connect worker의 출구 IP**가 필요합니다. 노트북 접속 성공만으로 worker 접속이 보장되지 않습니다.
클라우드 Kafka라면 Broker/REST 각각의 TLS·SASL·인증 정보를 해당 서비스 방식으로 적용하세요.

### 2. Worker 준비

1. Kafka Broker와 Kafka Connect distributed worker를 준비합니다.
2. MongoDB Kafka Connector 배포 JAR/패키지를 **모든 worker의 plugin.path**에 설치하고 worker를 재시작합니다.
3. worker의 `bootstrap.servers`와 내부 config/offset/status 토픽을 설정합니다.
   단일 Broker 실습은 내부 토픽 replication factor를 `1`로 설정합니다. 운영 값은 Broker 구성에 맞춥니다.
4. `GET /connector-plugins`에 `com.mongodb.kafka.connect.MongoSinkConnector`가 보이는지 확인합니다.
5. MongoDB URI는 worker의 ConfigProvider로 주입합니다. OpenCode MCP 설정은 Connect에 자동 전달되지 않습니다.

worker properties 예시(실제 worker 설정 파일에 추가):

```properties
# Kafka 표준 파일 ConfigProvider를 worker에서 활성화합니다.
config.providers=file
config.providers.file.class=org.apache.kafka.common.config.provider.FileConfigProvider
```

모든 worker에서 읽을 수 있는 `/run/secrets/mongodb.properties`에 `uri=<실제 URI>`를 저장하고
접근 권한을 제한합니다. 아래 Connector JSON은 `${file:/run/secrets/mongodb.properties:uri}`를 참조합니다.
이 경로는 **worker 파일시스템 경로**입니다. 노트북의 로컬 인증 파일 경로와 다릅니다.

### 3. REST 등록 순서

아래 `sink-config.json`은 각 Lab에서 제공하는 **config 객체만** 담은 파일입니다.

```bash
# plugin 설치 상태를 확인합니다.
curl --fail-with-body http://localhost:8083/connector-plugins

# 등록 전에 Connector 설정을 검증합니다. error_count=0인지 확인합니다.
curl --fail-with-body -X PUT \
  http://localhost:8083/connector-plugins/com.mongodb.kafka.connect.MongoSinkConnector/config/validate \
  -H 'Content-Type: application/json' --data-binary @sink-config.json

# 예시 이름 hrm-lab-sink로 Connector 설정을 등록/갱신합니다.
curl --fail-with-body -X PUT http://localhost:8083/connectors/hrm-lab-sink/config \
  -H 'Content-Type: application/json' --data-binary @sink-config.json

# Connector와 모든 task가 RUNNING인지 확인합니다.
curl --fail-with-body http://localhost:8083/connectors/hrm-lab-sink/status
```

`/status`의 task가 FAILED이면 trace와 worker 로그를 확인합니다. 설정 검증 성공은 실제 Atlas 접속 성공이 아닙니다.

### 4. 메시지 계약과 오류 처리

- `lab2/events.ejson`은 **JSON 배열**입니다. 배열 전체를 한 메시지로 보내지 말고 이벤트 객체 한 건씩 발행합니다.
- 발행 재시도에도 이벤트 `_id`를 유지합니다. Kafka key는 기기+리소스 조합으로 설정하여 같은 상태 키를 같은 partition에 보냅니다.
- partition 내부 전달 순서는 eventTime 순서와 다를 수 있습니다. 같은 시각 판정·부분 payload는 생산자 버전 계약이 필요합니다.
- Connect는 재시작/offset 재처리 시 같은 레코드를 재전달할 수 있습니다. 아래 기본 예시를 end-to-end exactly-once로 해석하지 않습니다.
- 기본 예시는 `errors.tolerance=none`으로 실패를 드러냅니다. 오류를 넘기려면 DLQ와 경보를 함께 구성합니다.
  단일 Broker 예시: `errors.tolerance=all`, `errors.deadletterqueue.topic.name=hrm.dlq`,
  `errors.deadletterqueue.topic.replication.factor=1`, `errors.deadletterqueue.context.headers.enable=true`.
  적용 후 정상 건수+DLQ 건수+offset/lag를 함께 확인하고 실패 원인 수정 후 재처리합니다.

### 5. 공통 확인표

| 관측 지점 | 확인 항목 |
|---|---|
| Producer | 발행 성공 callback/flush, 실제 발행 건수 |
| Kafka | topic/partition/offset, consumer group lag |
| Connect | Connector 및 task RUNNING, worker 로그, 실패/DLQ |
| Atlas | 대상 namespace, `_id`, BSON Date 타입, 건수·집계 |

`RUNNING`만으로 완료를 판정하지 말고 특정 이벤트 ID의 Atlas 도착까지 확인합니다.

공식 가이드:
[설치](https://www.mongodb.com/docs/kafka-connector/current/introduction/install/) ·
[전체 설정](https://www.mongodb.com/docs/kafka-connector/current/sink-connector/configuration-properties/all-properties/) ·
[ID 전략](https://www.mongodb.com/docs/kafka-connector/current/sink-connector/fundamentals/post-processors/) ·
[쓰기 전략](https://www.mongodb.com/docs/kafka-connector/current/sink-connector/fundamentals/write-strategies/)

## Lab 3과 Kafka 연결 — 원본 이력과 상태 처리

```text
Lab2 이벤트 → hrm.lab3.events ─┬─ MongoDB Sink → hrm_workshop_lab3.kafka_events
                              └─ 업무 Consumer → current_state → timer → anomaly_results
```

Sink는 원본 이력, Consumer는 최신 상태·조건부 갱신·timer를 담당합니다.
Consumer와 Sink는 서로 독립된 consumer group이므로 어느 쪽이 먼저 처리되는지는 보장되지 않습니다.
이 노트북의 `process_event()`는 이력도 저장하므로 Kafka로 확장할 때 **이력 소유자**를 정하세요.
예시 흐름에서는 Sink만 `kafka_events`를 쓰고, Consumer는 상태 갱신 로직을 별도 함수로 분리합니다.
Consumer 재시작 시 상태 처리 완료 후 offset을 commit하고, 재처리해도 같은 결과가 되도록 유지합니다.

### 일반 이력 Sink 설정 예시

```json
{
  "connector.class": "com.mongodb.kafka.connect.MongoSinkConnector",
  "topics": "hrm.lab3.events",
  "connection.uri": "${file:/run/secrets/mongodb.properties:uri}",
  "database": "hrm_workshop_lab3",
  "collection": "kafka_events",
  "tasks.max": "1",
  "key.converter": "org.apache.kafka.connect.storage.StringConverter",
  "value.converter": "org.apache.kafka.connect.storage.StringConverter",
  "document.id.strategy": "com.mongodb.kafka.connect.sink.processor.id.strategy.ProvidedInValueStrategy",
  "document.id.strategy.overwrite.existing": "false",
  "writemodel.strategy": "com.mongodb.kafka.connect.sink.writemodel.strategy.ReplaceOneDefaultStrategy",
  "bulk.write.ordered": "true",
  "errors.tolerance": "none"
}
```

value는 UTF-8 **Extended JSON 문자열**로 발행합니다. StringConverter는 문자열을 전달하고
MongoDB Connector가 JSON 문자열을 BSON으로 읽는 경로를 사용합니다. 원본 `$date`를 일반 중첩 객체로
취급하는 schemaless JsonConverter 예시와 혼용하지 마세요. 설치한 Connector 버전에서 실제 BSON Date를 검증합니다.
Avro/Protobuf/Schema Registry를 사용하는 환경은 해당 converter와 Date/Timestamp 논리 타입을 별도로 맞춥니다.

`ProvidedInValueStrategy`는 value의 `_id`를 사용합니다. ReplaceOne은 동일 ID를 upsert하므로
동일 payload 재발행 시 문서 수가 유지됩니다. **기존 원본을 불변으로 유지하는 `$setOnInsert`와는 다른 의미**이며
같은 ID로 다른 payload를 보내면 내용이 교체됩니다. 최초 값 보존이 필요하면 custom WriteModelStrategy나 consumer를 사용합니다.
`UpdateOneTimestampsStrategy`의 `_modifiedTS`는 처리 시각으로, eventTime 기반 최신 상태 판정을 대신하지 않습니다.

### 노트북 Producer 예시

먼저 선택 의존성 `confluent-kafka`를 설치합니다(`%pip install confluent-kafka`).
이 셀을 별도 실행할 때 Kafka 토픽을 준비하고 실제 bootstrap 주소를 적용합니다.

```python
# 현재 노트북에서 읽은 datetime을 Extended JSON $date로 직렬화합니다.
import json
from bson import json_util
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": "localhost:9092"})
delivery_errors = []

def delivered(error, message):
    # 비동기 발행 오류를 누락하지 않고 flush 후 확인합니다.
    if error is not None:
        delivery_errors.append(str(error))

for event in events:
    resource = event["resource"]
    key = json.dumps([event["deviceId"], resource["href"], resource["unitId"], resource["component"]])
    producer.produce("hrm.lab3.events", key=key.encode("utf-8"),
                     value=json_util.dumps(event).encode("utf-8"), on_delivery=delivered)
    producer.poll(0)
assert producer.flush(30) == 0 and not delivery_errors
```

### 단계별 검증

1. 새로운 `kafka_events` 컬렉션/전용 토픽으로 시작합니다. 기존 `events`의 직접 적재 결과와 섞지 않습니다.
2. 4,959건 발행 → Atlas 고유 ID 4,959건인지 확인합니다. lag가 0이 될 때까지 기다립니다.
3. 같은 이벤트 10건 재발행 → 이력 건수 유지, ID와 payload 동일 여부 확인.
4. OPEN→CLOSE→과거 OPEN 발행 → 업무 Consumer가 CLOSED를 유지하는지 확인.
5. OPEN만 발행 → **이벤트 수신과 독립된 timer tick**으로 180초 감지.
6. CLOSE 재처리 → 같은 열림 구간 알림 해소, consumer 재시작 후 중복 알림 없는지 확인.

```python
# Kafka 발행 후 연결 셀을 다시 실행해 종료된 client를 새로 연결합니다.
client, db = connect("hrm_workshop_lab3")
sample = db.kafka_events.find_one({"_id": events[0]["_id"]})
assert sample is not None and isinstance(sample["eventTime"], datetime)
assert db.kafka_events.count_documents({"datasetTag": "HRM_LAB2_DOOR_V1"}) == 4959
client.close()
```

### S3 이력과의 연결

Kafka→Atlas는 실시간 적재 경로입니다. 과거 JSON을 S3에 보관한다면 별도 S3 Sink/아카이빙 작업을 준비하고
Atlas Data Federation에서 Atlas와 S3 데이터 소스를 정의하여 조회할 수 있습니다.
중복 기간·eventId·날짜 타입·스키마를 맞춘 뒤 합산합니다. 이 Sink 설정만으로 S3 보관이 생성되지는 않습니다.